# Ejemplo LSA sobre descriptores TF-IDF

**Curso**: INF3841 - Recuperación de Información  
**Profesor**: Juan Manuel Barrios  
**Fecha**: 21 de septiembre de 2026


Primero se usará el mismo ejemplo de la semana anterior de crear descriptores tf-idf y calcular la similitud coseno entre todos.

Después se reducirá la dimensión de los vectores con LSA usando dos métodos:

 1. Factorizar las matrices con un método iterativo aproximado (más rápido)
 2. Factorizar las matrices con el método SVD tradicional (más lento)

Para cada uno se compara el resultado obtenido por LSA versus el obtenido por los descriptores originales.

Se usará el dataset `dataset_ciperchile.20-09-2026.zip` con artículos del sitio https://www.ciperchile.cl/  
Ver la descripción en su archivo `Readme.txt`.  
Dataset publicado solo para fines académicos.  
El derecho de autor de todos los textos pertenece a CIPER Chile. 

#### Calcular similitudes con tf-idf (igual a ejemplo semana anterior)


In [ ]:
import zipfile
import time
import numpy
import os


class File:
    def __init__(self):
        self.filename = ""
        self.titulo = ""
        self.autor = ""
        self.fecha = ""
        self.tipo = ""
        self.url = ""
        self.full_text = ""


class Dataset:
    def __init__(self, file_zip):
        self.dataset_files = []
        header = dict()
        print("leyendo {}...".format(file_zip))
        t1 = time.time()
        with zipfile.ZipFile(file_zip, mode="r") as zip:
            with zip.open("dataset.txt", mode="r") as f:
                for line in f:
                    fields = line.decode('utf-8').strip().split("\t")
                    # leer los nombres de la primera fila
                    if len(header) == 0:
                        header = dict((c, i) for i, c in enumerate(fields))
                        continue
                    file = File()
                    file.filename = fields[header["Filename"]]
                    file.titulo = fields[header["Título"]]
                    file.autor = fields[header["Autor"]]
                    file.fecha = fields[header["Fecha"]]
                    file.tipo = fields[header["Tipo_documento"]]
                    file.url = fields[header["URL"]]
                    self.dataset_files.append(file)
            print("leyendo {} archivos en {} ...".format(len(self.dataset_files), file_zip))
            for file in self.dataset_files:
                with zip.open("dataset/" + file.filename) as f:
                    full_text = f.read().decode('utf-8')
                    # quitar la primera linea (URL)
                    file.full_text = full_text[full_text.find("\n") + 1 :]
        t2 = time.time()
        print("dataset cargado en {:.1f} segundos".format(t2 - t1))


dataset = Dataset("dataset_ciperchile.20-09-2026.zip")

#### Seleccionar los documentos a usar (igual a ejemplo semana anterior)


In [ ]:
def fulltexts_documentos(dataset, num_documentos):
    textos = []
    documentos = []
    contador = 0
    largo_total = 0
    for doc in dataset.dataset_files[0:num_documentos]:
        textos.append(doc.full_text)
        documentos.append(doc)
        contador += 1
        largo_total += len(doc.full_text)
    print("Se leen {} documentos con largo total de {:.1f} MB".format(contador, largo_total / 1000 / 1000))
    return textos, documentos


# cantidad de documentos a leer, se puede reducir si es muy lento
num_documentos = len(dataset.dataset_files)     # un numero entre 1 y len(dataset.dataset_files)

# un array con los textos para crear el vocabulario y descriptores
textos, documentos = fulltexts_documentos(dataset, num_documentos)

#### Calcular vocabulario (igual a ejemplo semana anterior)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# calcular el vocabulario: total de palabras a representar
t0 = time.time()
vectorizer = TfidfVectorizer(
    analyzer="word", # definición de token o gram, por defecto "word" (dividir en palabras)
    lowercase=True,  # convertir a minusculas, por defecto es True
    strip_accents="unicode",  # por defecto es None. Usar "unicode" para eliminar acentos.
    sublinear_tf=True,  # por defecto es False. Usar True para 1+log(freq)
    use_idf=True,       # por defecto es True.
    norm="l2",          # por defecto es "l2", crea vectores unitarios y el coseno es solo la multiplicacion de valores.
    ngram_range=(1,1),  # rango de ngramas a usar por ej: (1,1) o (1,2) o (1,3). Por defecto es (1,1).
    max_df=1.0,  # Descartar palabras que aparecen en más de max_df documentos. Probar con 0.9.
                 #    -Si es int   -> cantidad de documentos
                 #    -Si es float -> porcentaje del total de documentos
                 #    -Notar 1 que es distinto a 1.0 (solo 1 documento vs el 100% de los documentos)
    min_df=0.0,  # Descartar palabras que aparecen en menos de min_df documentos (int o float). Probar con 0.1.
)
vectorizer.fit(textos)
t1 = time.time()

print("tiempo para crear vocabulario: {:.1f} segs".format(t1 - t0))
print("vocabulario de {} palabras".format(len(vectorizer.vocabulary_)))
print("primeras 10 palabras: ", list(vectorizer.vocabulary_.keys())[0:10])
print("primeras 10 palabras ordenadas: ", list(sorted(vectorizer.vocabulary_.keys())[0:10]))


#### Calcular descriptores (igual a ejemplo anterior)

In [ ]:
# transform() entrega el descriptor tf-idf (un vector sparse con los pesos de las palabras)
t0 = time.time()
descriptores = vectorizer.transform(textos)
t1 = time.time()

print("tiempo descriptores: {:.1f} segs".format(t1 - t0))
print("descriptores es una matriz de {}".format(descriptores.shape))

# verificar que los descriptores es una matriz sparse
print("descriptores es tipo {}".format(type(descriptores)))

#### Calcular similares (igual a ejemplo anterior)

In [ ]:
# usar la funcion toarray() para convertir los descriptores a una matriz densa
# multiplicar las matrices con matmul() para calcular la similitud coseno

t0 = time.time()
descriptores_denso = descriptores.toarray()
auto_similitud = numpy.matmul(descriptores_denso, descriptores_denso.T)
t1 = time.time()

print("Tiempo comparación todos contra todos: {:.1f} segs".format(t1 - t0))


#### Imprimir el resultado de similitud coseno (igual a ejemplo semana anterior)

In [ ]:
import pandas


def imprimir_similares(matriz, documentos, min_score):
    # obtener el maximo por fila
    # primero llenar la diagonal con ceros para que el mas parecido no sea si mismo
    numpy.fill_diagonal(matriz, 0)
    # recorrer las filas de la matriz
    filas = []
    for i in range(matriz.shape[0]):
        # obtener la posicion del maximo por fila
        posicion_mayor = numpy.argmax(matriz[i])
        # nombres de archivos
        documento = documentos[i]
        documento_mas_similar = documentos[posicion_mayor]
        similitud = matriz[i, posicion_mayor]
        # imprimir documentos con score mayor al mínimo pedido
        if similitud >= min_score:
            fila = {
                "query": documento.filename,
                "documento": documento_mas_similar.filename,
                "similitud coseno": similitud,
            }
            filas.append(fila)
    print("{} documentos con similitud mayor a {}:".format(len(filas), min_score))
    print()
    df = pandas.DataFrame(filas)
    print(df.to_string(index=False, justify="center"))


imprimir_similares(auto_similitud, documentos, min_score=0.0)

# Latent Semantic Analysis (LSA)

LSA consiste en reducir la cantidad de términos del vocabulario, agrupando los términos que aparecen correlacionados. Conceptualmente es muy similar a Análisis de Componentes Principales (PCA) que veremos más adelante en el curso.

A continuación se define la cantidad de conceptos que se van a calcular para reducir de dimensionalidad los vectores.

# LSA opción 1: usar método aproximado de Scikit-Learn

El algoritmo de TruncatedSVD aproxima la descomposición con un método iterativo (demora como 1 minuto).   

 * `n_components` define la nueva dimensionalidad de los vectores
 * `n_iter` es la cantidad de iteraciones del método aproximado. Al aumentarlo mejora la aproximación al resultado real aunque toma más tiempo de calcular.

Ver documentacion en: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html

In [ ]:
from sklearn.decomposition import TruncatedSVD

# numero de dimensiones a los que se reducirán los descriptores
# con un numero pequeño funciona todo muy rápido aunque no muy buena calidad
num_conceptos = 50

t0 = time.time()
transformer_tsvd = TruncatedSVD(n_components=num_conceptos, n_iter=20, random_state=4)
transformer_tsvd.fit(descriptores)
t1 = time.time()

print("tiempo ajustar la transformacion: {:.1f} segs".format(t1-t0))

## Revisar el espacio reducido y valores singulares

In [ ]:
#Se imprimen los valores singulares y su suma total con respecto al total
print("matriz proyeccion conceptos latentes={}".format(transformer_tsvd.components_.shape))
print("primer concepto latente={}".format(transformer_tsvd.components_[0]))
print("segundo concepto latente={}".format(transformer_tsvd.components_[1]))
print("tercer concepto latente={}".format(transformer_tsvd.components_[2]))
print("PORCENTAJE DE VARIANZA={:.1f}%".format(transformer_tsvd.explained_variance_ratio_.sum()*100))
print(transformer_tsvd.singular_values_)


## Mostrar los primeros conceptos latentes

In [ ]:
def imprimir_conceptos_tsvd(conceptos_a_mostrar, palabras_por_concepto):
    vocabulario = numpy.array(vectorizer.get_feature_names_out())
    for i, pesos in enumerate(transformer_tsvd.components_):
        if i >= conceptos_a_mostrar:
            break
        indices_menor_a_mayor = numpy.argsort(pesos)
        indices_mayor_a_menor = indices_menor_a_mayor[::-1]
        pesos_mayor_a_menor = pesos[indices_mayor_a_menor]
        terminos_mayor_a_menor = vocabulario[indices_mayor_a_menor]
        lista=[]
        for j in range(palabras_por_concepto):
            lista.append("{}({:.2f})".format(terminos_mayor_a_menor[j],pesos_mayor_a_menor[j]))
        print("CONCEPTO LATENTE #{:2d}: {}".format(i, " ".join(lista)))
        print()

imprimir_conceptos_tsvd(5, 10)

## Reducir de dimensión los descriptores tf-idf
Se obtiene la matriz de descriptores de dimension dada en el parámetro `n_components`.

In [ ]:
descriptores_tsvd = transformer_tsvd.transform(descriptores)
print("{} -> {} ".format(descriptores.shape, descriptores_tsvd.shape))

## Calcular la similitud en el espacio reducido

Como los descriptores tienen menos dimensiones, el tiempo de comparación es mucho menor al usado con los descriptores originales.

In [ ]:
t0 = time.time()
similitudes_tsvd = numpy.matmul(descriptores_tsvd, descriptores_tsvd.T)
t1 = time.time()
print("tiempo comparacion todos contra todos: {:.1f} segs".format(t1-t0))

## Comparar con las similitudes según los descriptores originales

In [ ]:
import pandas

def evaluar_similitud(matriz_similitud, conceptos, matriz_ideal, documentos):
    #completo la diagonal con cero para que el mas parecido no sea si mismo
    numpy.fill_diagonal(matriz_similitud, 0)
    numpy.fill_diagonal(matriz_ideal, 0)

    #imprimir los documentos mas parecidos y su similitud
    filas = []
    total = matriz_ideal.shape[0]
    total_iguales = 0
    for i in range(total):
        # obtener la posicion del maximo por fila
        posicion_mayor = numpy.argmax(matriz_similitud[i])
        posicion_mayor_ideal = numpy.argmax(matriz_ideal[i])
        # nombres de archivos
        documento = documentos[i]
        documentos_mas_similar = documentos[posicion_mayor]
        documentos_mas_similar_ideal = documentos[posicion_mayor_ideal]
        resultado = ""
        if posicion_mayor == posicion_mayor_ideal:
            resultado = "¡igual! :-) ok"
            total_iguales += 1
        else:
            resultado = "¡distinto! era {}".format(documentos_mas_similar_ideal.filename)
        fila = {
            "query": documento.filename,
            "documento": documentos_mas_similar.filename,
            "resultado": resultado,
        }
        filas.append(fila)
    pct_coincidencias = total_iguales / total * 100
    print("resultados al usar {} conceptos".format(conceptos))
    print("iguales al original: {} / {} = {:.1f}%".format(total_iguales, total, pct_coincidencias))
    print()
    df = pandas.DataFrame(filas)
    print(df.to_string(index=False, justify="center"))


In [ ]:
evaluar_similitud(similitudes_tsvd, descriptores_tsvd.shape[1], auto_similitud, documentos)

# LSA opción 2: Usar método exacto de factorización SVD

Usaremos la implementación de Numpy para la SVD (Singular Value Decomposition).  
Ver documentacion en: https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html

Es mucho más lento que el método anterior ya que es exacto. La ventaja es que es independiente del número de dimensiones a proyectar, por lo que se calcula una vez y luego se puede probar con distinta cantidad de conceptos latentes.

Demora entre 4 a 8 minutos (dependiendo del procesador).

In [ ]:
from numpy.linalg import svd

#demora unos 5 minutos para todo el dataset
t0 = time.time()
descriptores_denso = descriptores.toarray()
U, S, VT = numpy.linalg.svd(descriptores_denso, full_matrices=False, compute_uv=True)
t1 = time.time()
print("tiempo factorizacion USV: {:.1f} segs ({:.1f} mins)".format(t1-t0,(t1-t0)/60))

#notar que S se guarda como un array en vez de matriz
#con numpy.diag(S) obtenemos la matriz diagonal
print("A{} = U{} x S{} x V^T{}".format(descriptores_denso.shape, U.shape, S.shape, VT.shape))
print("array S: {} {} {} ... {} {}".format(S[0], S[1], S[3], S[-2], S[-1]))

## Seleccionar dimensiones

In [ ]:
# numero de dimensiones a los que se reducirán los descriptores
# notar que ya se realizó el cálculo de la factorización, por lo que ahora es directo probar con varios numeros
cantidad_conceptos = 50

#probar con distinto numero de conceptos (es rapido, no requiere volver a factorizar los descriptores)
Uk = U[:, :cantidad_conceptos]
Sk = S[0:cantidad_conceptos]
VkT = VT[:cantidad_conceptos, :]

print("Uk{} x Sk{} x Vk^T{}".format(Uk.shape, Sk.shape, VkT.shape))

#se imprimen los valores singulares y su suma total con respecto al total
print("PORCENTAJE DE VARIANZA={:.1f}%".format(numpy.sum(Sk)/numpy.sum(S)*100))
print(Sk)


In [ ]:
def imprimir_conceptos_svd(conceptos_a_mostrar, palabras_por_concepto):
    vocabulario = numpy.array(vectorizer.get_feature_names_out())
    for fila in range(conceptos_a_mostrar):
        pesos = numpy.abs(numpy.array(VkT[fila])) # pueden salir pesos negativos
        indices_menor_a_mayor = numpy.argsort(pesos)
        indices_mayor_a_menor = indices_menor_a_mayor[::-1]
        pesos_mayor_a_menor = pesos[indices_mayor_a_menor]
        terminos_mayor_a_menor = vocabulario[indices_mayor_a_menor]
        lista=[]
        for j in range(palabras_por_concepto):
            lista.append("{}({:.2f})".format(terminos_mayor_a_menor[j],pesos_mayor_a_menor[j]))
        print("CONCEPTO LATENTE #{:2d}: {}".format(fila, " ".join(lista)))
        print()

imprimir_conceptos_svd(5, 10)

## Reducir de dimensión los descriptores originales

In [ ]:
# METODO 1: reducir dimensiones de los descriptores 
Ak = numpy.matmul(descriptores_denso, VkT.T)
print("Ak={}".format(Ak.shape))

# METODO 2: alternativamente podemos obtener los descriptores reducidos multiplicando: U*S
#Esto se muestra como curiosidad, porque en la práctica uno deseará también transformar otras
#matrices de descriptores (como las queries) y eso solo se puede hacer con el método 1
Ak_alt = numpy.matmul(Uk, numpy.diag(Sk))
print("Ak_alt={}".format(Ak_alt.shape))

#comprobando la diferencia entre ambos metodos (es esencialmente 0, se comprueba que es lo mismo)
diferencias = numpy.subtract(Ak, Ak_alt)
print("diferencia entre Ak - Ak_alt: min={:.8f} max={:.8f}".format(numpy.min(diferencias), numpy.max(diferencias)))


## Calcular la similitud de los descriptores reducidos y comparar resultados con descriptores originales

In [ ]:
# comparar multiplicando matrices
t0 = time.time()
similitudes_svd= numpy.matmul(Ak, Ak.T)
t1 = time.time()
print("tiempo comparacion todos contra todos: {:.1f} segs".format(t1-t0))

evaluar_similitud(similitudes_svd, Ak.shape[1], auto_similitud, documentos)

# Ejercicios Propuestos

__1.__ Hacer un gráfico que muestre cómo aumenta la calidad de la respuesta al aumentar la cantidad de conceptos. ¿es un aumento lineal?


In [ ]:
# Hacer un ciclo con distintos número de conceptos. Para cada uno:
#  -Proyectar U, S y V a Uk, Sk y Vk (tomando las primeras columans)
#  -Reducir los descriptores originales (descriptores_denso) a Ak  (multiplicar matrices)
#  -Comparar los descriptores y encontrar la matriz de similityd
#  -Comparar con la matriz original llamando a evaluar_similitud()

__2.__ ¿Cómo buscar documentos similares cuando el documento de consulta no se usó para calcular LSA?

In [ ]:
#Calcular el documento más similar para los siguientes documentos de consulta.
textos_q = [ "elecciones de alcalde", "estudiantes de universidad"]

#1.calcular la matriz de descriptores para textos_q (usar el vocabulario ya calculado)
#2.proyectar la matriz de descriptores al espacio reducido
#3.buscar el documento más similar en el espacio reducido
#4.comprobar si se obtiene el mismo resultado en el espacio original
